# Clarification and reconnect

This notebook deliberately asks Epoch to confirm use of the two supplied years
before building. You answer with natural language and continue the same operation.
It also closes local observers while healthy server work continues.

Use **Epoch SDK (local prototype)**. This is a separate research operation from
notebook 01; its default ID is `notebook-demo-clarify`. Re-running a completed
operation retrieves its saved answer instead of asking the question again.

## 1. Connect and choose the readable operation ID

Run this cell again after any kernel restart. It reconstructs the client and operation ID without requiring objects from the old Python process.

The functions imported from `notebook_support.py` are walkthrough helpers.
`connect()` calls the SDK's `EpochClient.connect(...)` with your Epoch login and
both service addresses, then keeps the resulting client in the variable `epoch`.
`check_services()` prints the installed SDK versions and reads the services' health
endpoints. `await` means this cell waits for those network checks to finish.
This setup step prepares access; the later `epoch.research(...)` call starts research.


In [ ]:
# Import the SDK types and walkthrough helpers, then prepare a client.
# The same readable research ID lets later cells reconnect after a kernel restart.
import asyncio
from IPython.display import JSON, display
from epoch_sdk.research import ResearchCompleted
from epoch_sdk.supervisor import Answer, Answers, NeedsInput
from notebook_support import (
    connect, check_services, demo_request, operation_name,
    wait_research, wait_until_admitted, stop_local_observer,
    show_answer, check_demo_values,
)

# Create the SDK client using the Epoch login and service URLs supplied
# by the notebook server. Later cells use this object as "epoch".
epoch = connect()
RESEARCH_ID = operation_name("clarify")
request = demo_request(confirmation=True)
print("Research ID:", RESEARCH_ID)
# Print the installed SDK versions and check that Backend and EpochAI respond.
# "await" waits for those network checks to finish. Research starts in step 2.
await check_services()

## 2. Start research, then disconnect the original observer

Wait for the server to accept the native run before closing this Python wait. This calls no server-cancellation API. If the question already arrived, the local wait has simply finished.

In [ ]:
# An observer is this notebook waiting for the server to return a result.
# Start that wait as a background Python task so we can disconnect it.
# Wait for server acceptance before cancelling only the local wait.
# The server keeps the research operation; a fresh client can read its status.
observer = asyncio.create_task(epoch.research(request, operation_id=RESEARCH_ID))
admitted = await wait_until_admitted(epoch, RESEARCH_ID, observer)
cancelled = await stop_local_observer(observer)
await epoch.aclose()
print("Original local wait cancelled:", cancelled)
print("The research operation is still:", RESEARCH_ID)

## 3. Reconnect and read the pending question

For an actual kernel-restart test: choose **Kernel → Restart Kernel**, run step 1
again, then continue here. Skip step 2. A fresh client reads the same operation.
You can also run these cells normally to exercise a fresh client without a restart.

In [ ]:
# Replace the old Python client with a fresh one.
# Look up the same research ID and wait for its question or saved answer.
# feedback() formats any question and its options for you to read.
await epoch.aclose()
epoch = connect()
waiting = await wait_research(epoch, RESEARCH_ID)
if isinstance(waiting, NeedsInput):
    print(waiting.feedback())
    print("Thread:", waiting.handle.thread_id)
    print("Native run:", waiting.handle.run_id)
else:
    print("This operation already has its saved answer:", waiting.response.response_id)

## 4. Answer naturally, then disconnect after resume is accepted

Read the displayed question. The answer below is appropriate for this supplied-data
case; edit the string if needed. No code or SQL needs to be supplied as the answer.
The notebook converts your text into the native response format.

The demo expects one explicit confirmation question. If additional questions appear,
read them and provide an answer for each rather than applying one answer blindly.

In [ ]:
# This is your natural-language answer to the displayed confirmation question.
# The SDK needs the question ID and its position; the code below fills those in.
# resume_research() continues the existing thread from its waiting checkpoint.
# After the server accepts that continuation, stop this notebook waiting again.
ANSWER_TEXT = "Yes, use exactly the two supplied years: 2024 and 2025."

if isinstance(waiting, NeedsInput):
    assert len(waiting.interrupts) == 1, "Inspect and answer each returned interrupt."
    interrupt = waiting.interrupts[0]
    assert len(interrupt.ask_user().questions) == 1, "Inspect and answer each returned question."
    responses = {
        interrupt.id: Answers(answers=[
            Answer(question_index=0, free_form_input=ANSWER_TEXT)
        ])
    }
    prior_run = waiting.handle.run_id
    resumed_observer = asyncio.create_task(epoch.resume_research(waiting, responses))
    resumed = await wait_until_admitted(
        epoch, RESEARCH_ID, resumed_observer, previous_run_id=prior_run,
    )
    cancelled = await stop_local_observer(resumed_observer)
    await epoch.aclose()
    print("Resumed local wait cancelled:", cancelled)
    print("Server work continues under:", RESEARCH_ID)
else:
    print("Already completed; no question or run was submitted again.")

## 5. Reconnect to the saved answer

You may restart the kernel here as well: re-run step 1, then this step. Only the
readable ID is needed. The server publishes and pins the answer even if the original
and resumed clients never observe completion.

In [ ]:
# Create another fresh client and look up the same research ID.
# Wait for its saved answer, or show a further question if one is pending.
# show_answer() displays the saved table, JSON and text parts.
await epoch.aclose()
epoch = connect()
completed = await wait_research(epoch, RESEARCH_ID)
if isinstance(completed, NeedsInput):
    print(completed.feedback())
assert isinstance(completed, ResearchCompleted), "Answer the displayed pending question before proceeding."
await show_answer(epoch, completed)

## 6. Check exact values and repeat the original request

Repeating identical inputs with this ID must return the same saved answer and revision. It does not submit another research turn.

In [ ]:
# Read every page of the saved table and check the known demonstration values.
# Then repeat the original task with its original research ID.
# The asserts check that this returns the same answer and revision.
rows = []
async for page in epoch.iter_response_table(
    completed.response.response_id, "annual_sales",
    version=completed.response.version, page_size=1,
):
    rows.extend(page.rows)
check_demo_values(completed, rows)

attached = await epoch.research(request, operation_id=RESEARCH_ID)
assert isinstance(attached, ResearchCompleted)
assert attached.response == completed.response
print("Original saved answer:", attached.response.response_id)
print("Pinned revision:", attached.response.version)

## 7. Finish

The servers remain available. Keep the ID to read this operation again later. When trying a different task, choose a new readable ID. The next step is repeating these same actions through MCP in Codex.

In [ ]:
# Close this notebook client. Keep the readable ID to reconnect later.
await epoch.aclose()